# 07 - K-Means clustering (unsupervised - mostly a learning exercise)

**Idea.** K-means is **unsupervised**: it ignores the labels and just groups rows into 4 blobs by proximity. To turn it into a "classifier" we map each cluster to the majority training label inside it.

**Good for this data:** almost nothing for prediction. Useful only to *understand* structure, or to generate cluster-based features for a real classifier.

**Bad for this data:** it never sees the labels, and the stages overlap heavily in feature space, so cluster purity is only ~40% (best class per cluster) vs the 25% chance line. As a classifier it is barely better than guessing. **Do not expect a competitive submission from this.**

**Expected CV-style macro-F1:** ~0.40 (poor, as expected)

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

# Notebooks live in day9/models/, data sits one level up in day9/
TRAIN_PATH = '../train.csv'
TEST_PATH  = '../test.csv'
OUT_DIR    = '../outputs'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# every column except the id and the label is a predictor
FEATURES = [c for c in train.columns if c not in ('id', 'sleep_stage')]
TARGET   = 'sleep_stage'
y = train[TARGET]
print('train', train.shape, '| test', test.shape, '| #features', len(FEATURES))
train.head()

train (9000, 23) | test (5000, 22) | #features 21


,id,eeg_delta_power,eeg_theta_power,eeg_alpha_power,eeg_sigma_power,eeg_beta_power,eeg_gamma_power,eeg_slow_osc_power,eeg_spectral_entropy,eeg_spindle_density,...,eog_movement_density,eog_amplitude,heart_rate_mean,heart_rate_variability,respiration_rate,respiration_variability,spo2_mean,body_movement_index,eog_burst_index,sleep_stage
0,0,-1.51474,1.40728,10.33510,-1.61350,3.73081,0.99850,1.85689,-3.24253,-1.27096,...,2.65567,1.98733,1.60184,-2.49794,-0.59521,1.71154,1.93342,1.57365,-1.36230,1
1,1,-0.28998,0.89706,1.62494,2.41580,-2.70265,-0.10131,-1.68955,0.01442,-2.87943,...,4.36423,0.09942,3.38567,-0.56386,2.16016,-4.32301,1.07270,-2.43903,-0.37271,2
2,2,3.35435,0.32987,-5.41547,2.38680,-2.90584,-2.93372,-3.11713,-0.04647,1.61782,...,-3.87561,-0.87681,-2.84480,5.08383,-4.60411,0.37967,-2.06913,2.67324,NaN,3
3,3,-1.44917,-0.04374,1.71560,-1.27770,-0.19007,2.21826,1.69621,0.39756,0.00534,...,1.41415,0.39275,0.55060,-2.12910,2.32790,0.78319,0.98233,1.53824,-0.25040,1
4,4,1.35898,-2.36720,-7.40779,5.31815,-2.55954,-5.13284,-5.26634,1.73985,1.04618,...,-0.55616,0.86588,-1.96343,4.30036,0.22130,-1.44020,1.35760,-3.07701,-1.04947,3


## Preprocessing
K-means is distance-based, so scale + impute, exactly like kNN.

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans

prep = make_prep = None
imp = SimpleImputer(strategy='median')
sc  = StandardScaler()
X = sc.fit_transform(imp.fit_transform(train[FEATURES]))
Xte = sc.transform(imp.transform(test[FEATURES]))
print('scaled+imputed:', X.shape)

scaled+imputed: (9000, 21)


## Fit clusters and map clusters -> labels
We fit 4 clusters, then label each cluster with the most common true stage among its training members. This is the standard "cluster then vote" trick to evaluate clustering against labels.

In [3]:
km = KMeans(n_clusters=4, n_init=10, random_state=42)
train_clusters = km.fit_predict(X)

# majority true label inside each cluster
cluster_to_label = (
    pd.DataFrame({'cluster': train_clusters, 'y': y})
    .groupby('cluster')['y']
    .agg(lambda s: s.value_counts().idxmax())
    .to_dict()
)
print('cluster -> stage map:', cluster_to_label)

# cluster purity = how dominant the majority class is in each cluster
purity = (pd.DataFrame({'cluster': train_clusters, 'y': y})
          .groupby('cluster')['y'].agg(lambda s: s.value_counts(normalize=True).max()))
print('cluster purity (chance = 0.25):'); print(purity.round(2))

cluster -> stage map: {0: 1, 1: 1, 2: 0, 3: 2}
cluster purity (chance = 0.25):
cluster
0    0.42
1    0.48
2    0.44
3    0.39
Name: y, dtype: float64


## "Accuracy" of the cluster->label classifier
This is an in-sample, optimistic estimate and still poor - it confirms clustering is the wrong approach for the prediction task.

In [4]:
from sklearn.metrics import f1_score
train_pred = pd.Series(train_clusters).map(cluster_to_label)
print('train macro-F1 (optimistic):', round(f1_score(y, train_pred, average='macro'), 4))

train macro-F1 (optimistic): 0.3611


## Create submission (for completeness only - expect a weak score)

In [5]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
test_clusters = km.predict(Xte)
test_pred = pd.Series(test_clusters).map(cluster_to_label).astype(int)
submission = pd.DataFrame({'id': test['id'], 'sleep_stage': test_pred.values})
path = os.path.join(OUT_DIR, 'kmeans_submission.csv')
submission.to_csv(path, index=False)
print('wrote', path, submission.shape)
submission.head()

wrote ../outputs/kmeans_submission.csv (5000, 2)


,id,sleep_stage
0,9000,1
1,9001,2
2,9002,1
3,9003,2
4,9004,1


## Takeaway
K-means is **useless as a predictor here**. Its real value would be as a *feature engineer*: feed `cluster id` or `distance-to-each-centroid` into CatBoost/SVM. Don't submit this expecting a good leaderboard spot.